In [12]:
import pandas as pd

In [13]:
! pip install faiss-cpu

In [14]:

! pip install langchain==0.1.16

from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

In [15]:
import faiss
import numpy as np
import re

index = faiss.IndexFlatL2(768)

np.random.seed(42)

context_data = np.random.random((100, 768)).astype('float32')

index.add(context_data)

print(f"Indexed {index.ntotal} context vectors.")

Indexed 100 context vectors.


In [16]:
def semantic_search(query_em, index, top_k=5):           
    distances, indices = index.search(query_em, top_k)
    return indices

In [17]:
query_em = np.random.random((1, 768)).astype('float')
retrieved_indices = semantic_search(query_em, index)
print(f"Retrieved document indices: {retrieved_indices}")

Retrieved document indices: [[26 38 11 78 12]]


In [18]:
! pip install "transformers[torch]"

In [19]:
from transformers import GPT2Tokenizer,GPT2LMHeadModel


In [20]:
import torch

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [21]:
! pip install torch

In [22]:
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

In [23]:
context_texts = [
    "Retrieval-Augmented Generation combines a retriever and generator.",
    "It reduces hallucinations by grounding answers in retrieved documents.",
    "Uses Dense Passage Retrieval for semantic search which helps in the extraction of similar text.",
    "Employs Fusion-in-Decoder and Fusion-in-Encoder techniques.These techniques are used to integrate context with user query",
    "Provides up-to-date and domain-specific responses."
]

prompt_template = PromptTemplate(
    input_variables=["question", "context"],
    template="Question: {question}\nContext: {context}\nAnswer:"
)

memory = ConversationBufferMemory(
    memory_key="chat_history", return_messages=False)

In [ ]:
import numpy as np
def chat(question):
    query_embedding = np.random.rand(1, 768).astype("float32")
    retrieved_indices = semantic_search(query_embedding, index)
    context_texts = [f"Document {i}" for i in retrieved_indices[0]]

    chat_history = memory.load_memory_variables({}).get("chat_history", "")

    prompt = prompt_template.format(
        chat_history=chat_history,
        question=question,
        context="\n".join(context_texts)
    )

    inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True).to(device)

    
    
    outputs = model.generate(
        inputs,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        attention_mask=inputs["attention_mask"]
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response[len(prompt):].strip()
    response = re.sub(r"[\r\n]+", " ", response)

    memory.chat_memory.add_user_message(question)
    memory.chat_memory.add_ai_message(response)

    return response

In [32]:
tokenizer.pad_token = tokenizer.eos_token

In [33]:
print(chat("What is Retrieval-Augmented Generation (RAG)?"))


AttributeError: 